# Car-free guide for bike owners in Helsinki

Interactive map that suggests the cheapest sensible way to Helsinki Central Railway Station for someone
who owns a bike: **bike**, **bike + metro/train** or **public transport**, depending on travel times,
a €3.50 fare and the live weather.

Run all cells from top to bottom. The map is saved as `docs/index.html`.

In [ ]:
import pathlib
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"
OUTPUT_DIRECTORY = NOTEBOOK_PATH / "docs"

In [ ]:
import ssl
import geopandas

# Same workaround as in the course lessons: Noppe cannot verify the WFS server's certificate
ssl._create_default_https_context = ssl._create_unverified_context

# Travel time grid to Helsinki Central Railway Station (250 m cells, EPSG:3067)
grid = geopandas.read_file(
    DATA_DIRECTORY
    / "helsinki_region_travel_times_to_railway_station"
    / "helsinki_region_travel_times_to_railway_station.gpkg"
)

# City of Helsinki open data WFS; layers are requested in EPSG:3879
WFS_BASE_URL = (
    "https://kartta.hel.fi/ws/geoserver/avoindata/wfs"
    "?service=wfs&version=2.0.0&request=GetFeature"
    "&srsName=EPSG:3879&typeName={layer:s}"
)


def read_wfs(layer):
    """Read one WFS layer and reproject it to the CRS of the travel time grid."""
    return (
        geopandas.read_file(WFS_BASE_URL.format(layer=f"avoindata:{layer}"))
        .set_crs("EPSG:3879", allow_override=True)
        .to_crs(grid.crs)
    )


# Stations (where a bike can be taken on board) and the rail lines themselves
metro_stations = read_wfs("Seutukartta_liikenne_metroasemat")
train_stations = read_wfs("Seutukartta_liikenne_juna_asema")
metro_lines = read_wfs("Seutukartta_liikenne_metro_rata")
train_lines = read_wfs("Seutukartta_liikenne_juna_rata")

# Quick check: number of features, geometry types and column names of every layer
for name, layer in {
    "grid": grid,
    "metro_stations": metro_stations,
    "train_stations": train_stations,
    "metro_lines": metro_lines,
    "train_lines": train_lines,
}.items():
    print(f"{name}: {len(layer)} features, {layer.geom_type.unique()}, columns: {layer.columns.tolist()}")

In [ ]:
import numpy
import pandas

# Cycling assumptions
CITY_ZONE_M = 2500        # first 2.5 km of a trip (about 10 min at 15 km/h) are ridden in city traffic
CITY_SPEED_KMH = 15       # cycling speed in city traffic
OUTER_SPEED_KMH = 20      # cycling speed for the rest of the trip
STATION_SPEED_KMH = 15    # cycling speed on the short ride to a station
DETOUR_FACTOR = 1.3       # streets are not straight: route length = straight-line distance x 1.3

# Stations: the WFS gives station areas (polygons), so use their centroids;
# one train station has no geometry and is dropped
metro_pts = metro_stations.loc[metro_stations.geometry.notna(), ["metroasema", "geometry"]].rename(
    columns={"metroasema": "station_name"})
metro_pts["station_type"] = "Metro"
train_pts = train_stations.loc[train_stations.geometry.notna(), ["asemapaikka", "geometry"]].rename(
    columns={"asemapaikka": "station_name"})
train_pts["station_type"] = "Train"
stations = pandas.concat([metro_pts, train_pts], ignore_index=True)
stations["geometry"] = stations.geometry.centroid
stations = geopandas.GeoDataFrame(stations, geometry="geometry", crs=grid.crs)

# Grid: remove cells with no data (-1)
grid = grid.loc[
    (grid["walk_d"] >= 0) & (grid["pt_r_t"] >= 0) & (grid["car_r_t"] >= 0)
].copy()

# Option 1, bike only: walking route distance ridden at two speeds
city_part_m = grid["walk_d"].clip(upper=CITY_ZONE_M)
outer_part_m = (grid["walk_d"] - CITY_ZONE_M).clip(lower=0)
grid["bike_t"] = (
    city_part_m / 1000 / CITY_SPEED_KMH * 60
    + outer_part_m / 1000 / OUTER_SPEED_KMH * 60
)

# Option 2, public transport only: rush-hour travel time from the matrix (column pt_r_t)

# Option 3, bike + metro/train, part 1: the rail part of the trip is the rush-hour
# public transport time from the grid cell the station lies in (spatial join "within")
stations = geopandas.sjoin(
    stations, grid[["pt_r_t", "geometry"]], how="inner", predicate="within"
).rename(columns={"pt_r_t": "station_pt_t"}).drop(columns="index_right")

# Option 3, part 2: nearest station to the centre of every grid cell (nearest-neighbour join)
cell_centroids = geopandas.GeoDataFrame(
    grid[["from_id"]].copy(), geometry=grid.geometry.centroid, crs=grid.crs
)
nearest = geopandas.sjoin_nearest(
    cell_centroids, stations, how="left", distance_col="station_dist_m"
)
nearest = nearest[~nearest.index.duplicated(keep="first")]   # keep one station if two are equally close

# Option 3, part 3: ride to the station + rail time from the station
grid["station_name"] = nearest["station_name"]
grid["station_type"] = nearest["station_type"]
grid["bike_to_station_t"] = nearest["station_dist_m"] * DETOUR_FACTOR / 1000 / STATION_SPEED_KMH * 60
grid["bike_rail_t"] = grid["bike_to_station_t"] + nearest["station_pt_t"]

# Checks: how many stations were used, value ranges, and that every cell found a station
print(f"Stations used: {len(stations)} ({stations['station_type'].value_counts().to_dict()})")
print(grid[["bike_t", "pt_r_t", "bike_to_station_t", "bike_rail_t"]].describe().round(1))
print("Cells without a nearest station:", grid["bike_rail_t"].isna().sum())

In [ ]:
# Decision rules
FARE_EUR = 3.5                 # price of a public transport trip
BIKE_EXTRA_MIN_ACCEPTED = 5    # up to 5 extra minutes on the bike are worth saving the fare
RAIN_MAX_BIKE_MIN = 15         # in rain, cycle only short trips

# Fastest paid option for every cell: public transport only, or bike + metro/train
paid = grid[["pt_r_t", "bike_rail_t"]].rename(
    columns={"pt_r_t": "Public transport", "bike_rail_t": "Bike + metro/train"})
grid["paid_t"] = paid.min(axis=1)
grid["paid_mode"] = paid.idxmin(axis=1)

# Normal weather: bike, unless a paid option saves more than 5 minutes
grid["rec_normal"] = numpy.where(
    grid["bike_t"] <= grid["paid_t"] + BIKE_EXTRA_MIN_ACCEPTED, "Bike", grid["paid_mode"])

# Rain: bike only for short trips, otherwise the fastest paid option
grid["rec_rain"] = numpy.where(grid["bike_t"] <= RAIN_MAX_BIKE_MIN, "Bike", grid["paid_mode"])

# Snowing now, heavy snow in the last 8 hours, or colder than -15 °C: no cycling at all
grid["rec_snow_cold"] = "Public transport"

# Trip time and fare of the recommended option, for each weather situation
MODE_TIME = {"Bike": "bike_t", "Public transport": "pt_r_t", "Bike + metro/train": "bike_rail_t"}
SCENARIOS = {
    "normal": "Normal weather",
    "rain": "Rain",
    "snow_cold": "Snow or extreme cold (no cycling)",
}
for key in SCENARIOS:
    rec = grid[f"rec_{key}"]
    grid[f"rec_t_{key}"] = numpy.select(
        [rec == mode for mode in MODE_TIME], [grid[col] for col in MODE_TIME.values()])
    grid[f"fare_{key}"] = numpy.where(rec == "Bike", 0.0, FARE_EUR)

# Check: share of grid cells per recommendation in each weather situation
for key, name in SCENARIOS.items():
    print(f"\n{name}:")
    print(grid[f"rec_{key}"].value_counts(normalize=True).mul(100).round(1).to_string())

In [ ]:
import json
import urllib.request
from datetime import datetime

# Weather thresholds
HEAVY_SNOW_CM_8H = 5     # this much snow in the last 8 hours makes bike lanes hard to use
COLD_LIMIT_C = -15       # below this temperature, no cycling

# Open-Meteo (free, no API key): current weather, snowfall of the last 8 hours,
# and rain/snow forecast for the next 2 hours in 15-minute steps, for central Helsinki
WEATHER_URL = (
    "https://api.open-meteo.com/v1/forecast?latitude=60.17&longitude=24.94"
    "&current=temperature_2m,precipitation,rain,showers,snowfall"
    "&hourly=snowfall&past_hours=8&forecast_hours=1"
    "&minutely_15=rain,snowfall&forecast_minutely_15=8"
    "&timezone=Europe%2FHelsinki"
)


def upcoming_precipitation(minutely, now):
    """Return e.g. 'Rain in ~20 min' for the first 15-minute step with rain or snow, else None."""
    for time, rain, snow in zip(minutely["time"], minutely["rain"], minutely["snowfall"]):
        minutes = (datetime.fromisoformat(time) - now).total_seconds() / 60
        if minutes < 0:
            continue
        minutes = max(5, round(minutes / 5) * 5)
        if (snow or 0) > 0:
            return f"Snow in ~{minutes} min"
        if (rain or 0) > 0:
            return f"Rain in ~{minutes} min"
    return None


def weather_status(temp_c, wet_mm, snowfall_cm, snow_8h_cm, upcoming):
    """Apply the weather rules in order of priority; return the map layer and a short label."""
    if temp_c < COLD_LIMIT_C:
        return "snow_cold", "Too cold to bike"
    if snowfall_cm > 0:
        return "snow_cold", "Snowing"
    if snow_8h_cm >= HEAVY_SNOW_CM_8H:
        return "snow_cold", "Snowy bike lanes"
    if wet_mm > 0:
        return "rain", "Raining"
    if upcoming:
        return "normal", upcoming
    return "normal", "Normal"


# Weather when the map is made; the saved map updates this live when it is opened
try:
    with urllib.request.urlopen(WEATHER_URL, timeout=15) as response:
        weather = json.load(response)
    current = weather["current"]
    temp_c = current.get("temperature_2m") or 0
    rain_mm = (current.get("rain") or 0) + (current.get("showers") or 0)
    wet_mm = max(rain_mm, current.get("precipitation") or 0)
    snowfall_cm = current.get("snowfall") or 0
    snow_8h_cm = sum(v or 0 for v in weather["hourly"]["snowfall"])
    upcoming = upcoming_precipitation(weather["minutely_15"], datetime.fromisoformat(current["time"]))
    initial_key, weather_label = weather_status(temp_c, wet_mm, snowfall_cm, snow_8h_cm, upcoming)
    weather_chip = f"{temp_c:.0f} °C · {weather_label}"
except Exception as error:
    # Without weather data, start with normal weather
    initial_key = "normal"
    weather_chip = "Weather unavailable"
    print("Weather error:", error)

print(initial_key, "|", weather_chip)

In [ ]:
import folium
import xyzservices.providers as xyz
from folium.plugins import Geocoder, LocateControl
from jinja2 import Template

# Colours: one per recommended mode (colour-blind friendly qualitative scheme)
MODE_COLOURS = {
    "Bike": "#1b9e77",
    "Bike + metro/train": "#d95f02",
    "Public transport": "#7570b3",
}
STATION_COLOURS = {"Metro": "#e41a1c", "Train": "#444444"}


class PinGeocoder(Geocoder):
    """Search bar (OpenStreetMap Nominatim) that hands the found place to the pin code below
    instead of adding its own marker. Results are limited to the Helsinki region."""
    _template = Template("""
        {% macro script(this, kwargs) %}
        var {{ this.get_name() }} = L.Control.geocoder({
            position: "topleft",
            collapsed: false,
            defaultMarkGeocode: false,
            placeholder: "Search a place…",
            geocoder: L.Control.Geocoder.nominatim({
                geocodingQueryParams: {countrycodes: "fi", viewbox: "24.5,60.45,25.3,60.0", bounded: 1}
            })
        }).addTo({{ this._parent.get_name() }});
        {% endmacro %}
    """)


# Grid for the web map: WGS84, coordinates rounded to ~1 m to keep the HTML file smaller
web = grid.to_crs("EPSG:4326")
web["geometry"] = web.geometry.set_precision(0.00001)
web["bike_min"] = web["bike_t"].round().astype(int)
web["pt_min"] = web["pt_r_t"].astype(int)
web["bike_rail_min"] = web["bike_rail_t"].round().astype(int)
web["nearest_station"] = web["station_name"].fillna("") + " (" + web["station_type"].fillna("") + ")"

# Base map: light grey, so the coloured cells stand out (always on, not in the layer control)
interactive_map = folium.Map(tiles=None, control_scale=True)
folium.TileLayer(xyz.Esri.WorldGrayCanvas, control=False).add_to(interactive_map)

# One layer per weather situation. The weather decides which one is shown,
# so these layers are hidden from the layer control.
weather_layers = {}
for key in SCENARIOS:
    layer_data = web[["bike_min", "pt_min", "bike_rail_min", "nearest_station", "geometry"]].copy()
    layer_data["recommended"] = web[f"rec_{key}"]
    trip_min = web[f"rec_t_{key}"].round().astype(int).astype(str)
    fare = web[f"fare_{key}"].map(lambda f: "free" if f == 0 else f"{f:.2f} €")
    layer_data["tip"] = layer_data["recommended"] + " · " + trip_min + " min · " + fare  # hover text

    group = folium.FeatureGroup(name=SCENARIOS[key], control=False, show=(key == initial_key))
    folium.GeoJson(
        layer_data,
        style_function=lambda feature: {
            "fillColor": MODE_COLOURS[feature["properties"]["recommended"]],
            "color": "transparent",
            "weight": 0,
            "fillOpacity": 0.6,
        },
        highlight_function=lambda feature: {"color": "black", "weight": 1.5},
        tooltip=folium.GeoJsonTooltip(fields=["tip"], labels=False),       # hover: suggestion only
        popup=folium.GeoJsonPopup(                                          # click: all travel times
            fields=["bike_min", "pt_min", "bike_rail_min", "nearest_station"],
            aliases=["Bike only (min)", "Public transport (min)", "Bike + metro/train (min)",
                     "Nearest station"],
        ),
    ).add_to(group)
    group.add_to(interactive_map)
    weather_layers[key] = group

# Metro and train lines
lines = folium.FeatureGroup(name="Metro and train lines")
folium.GeoJson(
    metro_lines.to_crs("EPSG:4326")[["geometry"]],
    style_function=lambda feature: {"color": STATION_COLOURS["Metro"], "weight": 2.5},
).add_to(lines)
folium.GeoJson(
    train_lines.to_crs("EPSG:4326")[["geometry"]],
    style_function=lambda feature: {"color": STATION_COLOURS["Train"], "weight": 1.5},
).add_to(lines)
lines.add_to(interactive_map)

# Stations with a tooltip (name) and a pop-up (rail time to the centre)
station_group = folium.FeatureGroup(name="Stations (bike allowed on board)")
for _, row in stations.to_crs("EPSG:4326").iterrows():
    folium.CircleMarker(
        location=(row.geometry.y, row.geometry.x),
        radius=4,
        color="white",
        weight=1,
        fill=True,
        fill_color=STATION_COLOURS[row["station_type"]],
        fill_opacity=1,
        tooltip=row["station_name"],
        popup=folium.Popup(
            f"<b>{row['station_name']}</b> ({row['station_type']})<br>"
            f"To Central Railway Station: {int(row['station_pt_t'])} min<br>"
            "Bike allowed on board if there is space",
            max_width=220,
        ),
    ).add_to(station_group)
station_group.add_to(interactive_map)

# Destination of all trips: Helsinki Central Railway Station
destination = grid.loc[grid["from_id"] == grid["to_id"]]
if not destination.empty:
    point = destination.geometry.centroid.to_crs("EPSG:4326").iloc[0]
    folium.Marker(
        location=(point.y, point.x),
        tooltip="Helsinki Central Railway Station",
        icon=folium.Icon(color="black", icon="star"),
    ).add_to(interactive_map)

# Controls: zoom to the user's location when the map opens, search bar, layer control
LocateControl(
    auto_start=True,
    strings={"title": "Show my location"},
    locateOptions={"maxZoom": 14},
).add_to(interactive_map)
search_bar = PinGeocoder()
search_bar.add_to(interactive_map)
folium.LayerControl(collapsed=True).add_to(interactive_map)

# North arrow (top right, below the layer control)
north_arrow_html = """
<div title="North" style="position: fixed; top: 62px; right: 12px; z-index: 900; width: 30px;
     padding: 3px 0; background: rgba(255,255,255,0.93); border-radius: 6px;
     box-shadow: 0 1px 4px rgba(0,0,0,0.3); text-align: center; font: 700 12px/1.1 sans-serif;">
  <div style="font-size: 15px;">&#9650;</div>N
</div>
"""
interactive_map.get_root().html.add_child(folium.Element(north_arrow_html))

# Collapsible legend (bottom right): a small button that opens colours, fares and rules
swatch = "display:inline-block;width:10px;height:10px;border-radius:2px;margin-right:5px;"
legend_html = f"""
<details style="position: fixed; bottom: 40px; right: 10px; z-index: 9999;
                background: rgba(255,255,255,0.93); padding: 5px 9px; border-radius: 6px;
                box-shadow: 0 1px 4px rgba(0,0,0,0.3); font: 12px/1.55 sans-serif; max-width: 250px;">
  <summary style="cursor:pointer; font-weight:600;">Legend</summary>
  <div style="margin-top:4px;">
    <b>Best way to the city centre</b><br>
    <span style="{swatch}background:{MODE_COLOURS['Bike']};"></span>Bike · free<br>
    <span style="{swatch}background:{MODE_COLOURS['Bike + metro/train']};"></span>Bike + metro/train · {FARE_EUR:.2f} €<br>
    <span style="{swatch}background:{MODE_COLOURS['Public transport']};"></span>Public transport · {FARE_EUR:.2f} €
    <div style="font-size:11px; line-height:1.4; margin-top:6px; color:#333;">
      Trips to Helsinki Central Railway Station at rush hour. Search a place, or right-click / long-press
      to drop a pin, to get the suggestion there. Click a cell for all travel times.<br><br>
      <b>Normal:</b> bike, unless a paid option saves more than {BIKE_EXTRA_MIN_ACCEPTED} min.<br>
      <b>Rain:</b> bike only if the ride is {RAIN_MAX_BIKE_MIN} min or shorter.<br>
      <b>Snowing, ≥{HEAVY_SNOW_CM_8H} cm snow in 8 h, or below {COLD_LIMIT_C} °C:</b> no cycling.<br><br>
      Bike: {CITY_SPEED_KMH} km/h first {CITY_ZONE_M / 1000:g} km, then {OUTER_SPEED_KMH} km/h;
      to station: straight line × {DETOUR_FACTOR} at {STATION_SPEED_KMH} km/h.
      Bikes fit on trains and metro at rush hour only if there is space.
    </div>
  </div>
</details>
"""
interactive_map.get_root().html.add_child(folium.Element(legend_html))

# Weather chip (top centre): filled in when the map was made, updated live when opened
weather_box_html = f"""
<div id="weather-box" style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
     z-index: 9999; background: rgba(255,255,255,0.93); padding: 4px 12px; border-radius: 14px;
     box-shadow: 0 1px 4px rgba(0,0,0,0.3); font: 600 13px/1.4 sans-serif; white-space: nowrap;">
  {weather_chip}
</div>
"""
interactive_map.get_root().html.add_child(folium.Element(weather_box_html))

# JavaScript, run in the browser when the map opens: add data sources to the attribution,
# fetch the current weather from Open-Meteo, apply the same rules as in Python,
# show the matching weather layer and update the weather chip
weather_js = """
document.addEventListener("DOMContentLoaded", function () {
  var map = __MAP__;
  map.attributionControl.addAttribution(
    "Travel times: Helsinki Region Travel Time Matrix (Digital Geography Lab, Univ. of Helsinki) | "
    + "Lines & stations: City of Helsinki | Weather: Open-Meteo.com");

  var layers = {normal: __NORMAL__, rain: __RAIN__, snow_cold: __SNOW_COLD__};
  var box = document.getElementById("weather-box");
  var num = function (v) { return (typeof v === "number" && !isNaN(v)) ? v : 0; };

  // Show one weather layer and hide the others
  function showLayer(key) {
    Object.keys(layers).forEach(function (k) {
      if (map.hasLayer(layers[k])) { map.removeLayer(layers[k]); }
    });
    map.addLayer(layers[key]);
  }

  // First 15-minute step with rain or snow in the next 2 hours, e.g. "Rain in ~20 min"
  function upcoming(data) {
    var m = data.minutely_15 || {};
    var times = m.time || [];
    var offset = num(data.utc_offset_seconds) * 1000;
    for (var i = 0; i < times.length; i++) {
      var minutes = (Date.parse(times[i] + "Z") - offset - Date.now()) / 60000;
      if (minutes < 0) { continue; }
      minutes = Math.max(5, Math.round(minutes / 5) * 5);
      if (num((m.snowfall || [])[i]) > 0) { return "Snow in ~" + minutes + " min"; }
      if (num((m.rain || [])[i]) > 0) { return "Rain in ~" + minutes + " min"; }
    }
    return null;
  }

  fetch("__URL__")
    .then(function (response) {
      if (!response.ok) { throw new Error("HTTP " + response.status); }
      return response.json();
    })
    .then(function (data) {
      var c = data.current || {};
      var temp = num(c.temperature_2m);
      var wet = Math.max(num(c.rain) + num(c.showers), num(c.precipitation));
      var snowNow = num(c.snowfall);
      var snow8h = ((data.hourly || {}).snowfall || []).reduce(function (a, b) { return a + num(b); }, 0);
      var soon = upcoming(data);
      var key, label;
      if (temp < __COLD__) { key = "snow_cold"; label = "Too cold to bike"; }
      else if (snowNow > 0) { key = "snow_cold"; label = "Snowing"; }
      else if (snow8h >= __HEAVY__) { key = "snow_cold"; label = "Snowy bike lanes"; }
      else if (wet > 0) { key = "rain"; label = "Raining"; }
      else if (soon) { key = "normal"; label = soon; }
      else { key = "normal"; label = "Normal"; }
      showLayer(key);
      box.textContent = Math.round(temp) + " °C · " + label;
    })
    .catch(function (error) {
      // Keep the weather from when the map was made
      console.error("Live weather failed:", error);
    });
});
"""
weather_js = (
    weather_js
    .replace("__MAP__", interactive_map.get_name())
    .replace("__NORMAL__", weather_layers["normal"].get_name())
    .replace("__RAIN__", weather_layers["rain"].get_name())
    .replace("__SNOW_COLD__", weather_layers["snow_cold"].get_name())
    .replace("__URL__", WEATHER_URL)
    .replace("__COLD__", str(COLD_LIMIT_C))
    .replace("__HEAVY__", str(HEAVY_SNOW_CM_8H))
)
interactive_map.get_root().script.add_child(folium.Element(weather_js))

# JavaScript for pins: a search result, a right-click / long-press, or the user's location
# gets a pop-up with the suggestion of the grid cell under it, and the map centres there
pin_js = """
document.addEventListener("DOMContentLoaded", function () {
  var map = __MAP__;
  var searchBar = __SEARCH__;
  var weatherGroups = [__NORMAL__, __RAIN__, __SNOW_COLD__];
  var pin = null;

  // Find the grid cell under a point in the weather layer that is currently shown
  function suggestionAt(latlng) {
    var text = "Outside the mapped area";
    weatherGroups.forEach(function (group) {
      if (!map.hasLayer(group)) { return; }
      group.eachLayer(function (geojson) {
        if (!geojson.eachLayer) { return; }
        geojson.eachLayer(function (cell) {
          if (cell.getBounds && cell.getBounds().contains(latlng)) {
            text = cell.feature.properties.tip;
          }
        });
      });
    });
    return text;
  }

  // Replace the previous pin with a new one, open its pop-up and centre the map on it
  function dropPin(latlng, label) {
    if (pin) { map.removeLayer(pin); }
    pin = L.marker(latlng).addTo(map);
    pin.bindPopup("<b>" + label + "</b><br>" + suggestionAt(latlng)).openPopup();
    map.setView(latlng, Math.max(map.getZoom(), 14));
  }

  searchBar.on("markgeocode", function (e) {
    dropPin(e.geocode.center, e.geocode.name.split(",")[0]);
  });
  map.on("contextmenu", function (e) { dropPin(e.latlng, "Dropped pin"); });
  map.on("locationfound", function (e) {
    L.popup().setLatLng(e.latlng)
      .setContent("<b>You are here</b><br>" + suggestionAt(e.latlng))
      .openOn(map);
  });
});
"""
pin_js = (
    pin_js
    .replace("__MAP__", interactive_map.get_name())
    .replace("__SEARCH__", search_bar.get_name())
    .replace("__NORMAL__", weather_layers["normal"].get_name())
    .replace("__RAIN__", weather_layers["rain"].get_name())
    .replace("__SNOW_COLD__", weather_layers["snow_cold"].get_name())
)
interactive_map.get_root().script.add_child(folium.Element(pin_js))

# Start view: the whole study area (the location control zooms to the user if allowed)
minx, miny, maxx, maxy = web.total_bounds
interactive_map.fit_bounds([[miny, minx], [maxy, maxx]])

# Save the map as HTML in the docs folder
OUTPUT_DIRECTORY.mkdir(exist_ok=True)
output_file = OUTPUT_DIRECTORY / "index.html"
interactive_map.save(output_file)
print(f"Saved {output_file.name}: {output_file.stat().st_size / 1e6:.1f} MB")

interactive_map

## A car-free guide for bike owners in Helsinki

### Question
For someone who owns a bike and wants to spend as little as possible on getting around, what is the
most sensible way to get from any place in the Helsinki region to the city centre (Helsinki Central
Railway Station), and how does the answer change with Finnish weather? I commute around Helsinki by
electric bike, so this is a question I face every day.

### Data
- **Helsinki Region Travel Time Matrix**, travel times to Helsinki Central Railway Station (Digital
  Geography Lab, University of Helsinki), the same file as in Problem 1: walking route distance
  (`walk_d`) and rush-hour public transport time (`pt_r_t`).
- **Metro stations, train stations, metro lines and train lines** from the City of Helsinki WFS
  (`Seutukartta_liikenne_metroasemat`, `_juna_asema`, `_metro_rata`, `_juna_rata`).
- **Weather** from the Open-Meteo API (CC BY 4.0): current temperature, rain and snowfall, snowfall
  of the last 8 hours, and a 2-hour forecast in 15-minute steps.
- **Place search** with OpenStreetMap Nominatim; **base map** Esri World Gray Canvas.

### Analysis
1. **Three car-free options for every grid cell.**
   - *Bike only:* estimated from the walking route distance, 15 km/h for the first 2.5 km
     (city traffic) and 20 km/h after that, as in Problem 1.
   - *Public transport only:* rush-hour travel time from the matrix.
   - *Bike + metro/train:* in the HSL area a bike can be taken on the metro and commuter trains
     (not on buses or trams). I found the nearest station for every cell with a nearest-neighbour
     join (`sjoin_nearest`). The ride to the station is the straight-line distance × 1.3 at 15 km/h,
     and the rail part is the rush-hour public transport time from the grid cell the station lies in
     (spatial join `within`).
2. **Cost rule:** cycling is free and a public transport trip costs €3.50. Up to 5 extra minutes on
   the bike are worth saving the fare, so the bike is recommended unless a paid option is more than
   5 minutes faster.
3. **Weather rules**, one map layer each:
   - *Normal weather:* the cost rule above.
   - *Rain:* bike only if the ride takes 15 minutes or less, otherwise the fastest paid option.
   - *Snow or extreme cold:* no cycling (public transport only) when it is snowing, when at least
     5 cm of snow has fallen in the last 8 hours (bike lanes may not be cleared), or when it is
     colder than −15 °C.

### Map design
- **Automatic weather:** when the map is opened, it fetches the current Helsinki weather and shows
  the matching layer. A small chip at the top shows only the temperature and one word (e.g.
  "Normal", "Raining", "Rain in ~20 min"). If the live weather cannot be loaded, the map keeps the
  weather from when it was made.
- **Three colours, one per recommended mode** (qualitative, colour-blind friendly scheme), on a quiet
  grey base map.
- **Interaction:** hovering over a cell shows only the suggestion ("Bike · 18 min · free"); clicking
  shows all travel times and the nearest station. Stations have tooltips and pop-ups. Metro and train
  lines and stations can be switched on and off in the layer control.
- **Search and pins:** a search bar (geocoding, as in lesson 3), right-click / long-press to drop a pin,
  and the user's location when the map opens. Each one centres the map and shows the suggestion
  for that spot.
- **Minimal layout:** a collapsible legend with the rules, a north arrow, a scale bar, and the data
  sources in the attribution line.

### Result
In normal weather, bike + metro/train is the best car-free choice in 46 % of the grid cells, the bike
alone in 32 % and public transport alone in 22 %. The bike wins in and around the centre, and
bike + metro/train along the rail corridors. In rain, the bike alone is recommended in only 3 % of
cells, and bike + metro/train (66 %) becomes the main option. With snow or extreme cold, public
transport is the only recommendation. For a bike owner, combining the bike with the metro or train
is therefore often the most sensible way to the centre.

### Limitations
- All suggestions are for trips **to Helsinki Central Railway Station**, because the travel time data
  only covers that destination. Trips between any two places would need a routing service such as
  HSL's Digitransit API, which would be a possible extension.
- Bike times are estimates with fixed speeds; an electric bike would usually be faster.
- The rail time from a station's grid cell includes walking and waiting from the centre of that cell,
  so the bike + metro/train time is an approximation.
- Bikes fit on trains and metro at rush hour only if there is space.
- The weather is measured at one point in central Helsinki and used for the whole region.
- The €3.50 fare assumes single tickets; with a season ticket, the cost rule would change.